In [2]:
import torch
import torch.nn as nn
from torchvision import transforms
from src.gtsrb import NUM_CLASSES, get_dataloaders, save_predictions

In [3]:
train_augmentation = transforms.Compose([

    transforms.RandomRotation(25),

    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.25,
        saturation=0.2
    ),

    transforms.RandomResizedCrop(
        size=32
    ),

    transforms.ToTensor(),
    
    transforms.Normalize(
        mean=[0.3403, 0.3121, 0.3214],
        std=[0.2724, 0.2608, 0.2669]
    )

])

In [4]:
train_loader, val_loader, test_loader = get_dataloaders(img_size=32, batch_size=128, train_transform=train_augmentation)

In [5]:
def train_model(optimizer_name):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = nn.Sequential(

        nn.Conv2d(3, 32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Flatten(),

        nn.Linear(64 * 8 * 8, 128),
        nn.ReLU(),

        nn.Linear(128, NUM_CLASSES)

    ).to(device)

    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "SGD":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=0.001
        )

    elif optimizer_name == "Momentum":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=0.001,
            momentum=0.9
        )

    elif optimizer_name == "Adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=0.001
        )

    epochs = 5

    losses = []
    accuracies = []

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss

        losses.append(epoch_loss)

        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                predictions = outputs.argmax(dim=1)

                correct += (
                    predictions == labels
                ).sum().item()

                total += labels.size(0)

        accuracy = 100 * correct / total

        accuracies.append(accuracy)

        print(
            f"{optimizer_name} | "
            f"Epoch {epoch+1} | "
            f"Loss: {epoch_loss:.4f} | "
            f"Validation Accuracy: {accuracy:.2f}%"
        )

    return model, losses, accuracies

In [6]:
def generate_predictions(model):

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    model.eval()

    predictions_list = []

    with torch.no_grad():

        for images, _ in test_loader:

            images = images.to(device)

            outputs = model(images)

            predictions = outputs.argmax(dim=1)

            predictions_list.extend(
                predictions.cpu().numpy()
            )

    return predictions_list

In [7]:
model_sgd, losses_sgd, accs_sgd = train_model("SGD")

c:\Users\rafae\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


SGD | Epoch 1 | Loss: 603.8944 | Validation Accuracy: 8.28%
SGD | Epoch 2 | Loss: 575.0788 | Validation Accuracy: 13.12%
SGD | Epoch 3 | Loss: 557.5040 | Validation Accuracy: 16.55%
SGD | Epoch 4 | Loss: 542.6769 | Validation Accuracy: 18.54%
SGD | Epoch 5 | Loss: 529.9875 | Validation Accuracy: 21.38%


In [8]:
model_momentum, losses_momentum, accs_momentum = train_model("Momentum")

Momentum | Epoch 1 | Loss: 548.3871 | Validation Accuracy: 28.96%
Momentum | Epoch 2 | Loss: 464.9137 | Validation Accuracy: 36.34%
Momentum | Epoch 3 | Loss: 423.3818 | Validation Accuracy: 40.03%
Momentum | Epoch 4 | Loss: 395.4420 | Validation Accuracy: 46.83%
Momentum | Epoch 5 | Loss: 373.4312 | Validation Accuracy: 52.01%


In [9]:
model_adam, losses_adam, accs_adam = train_model("Adam")

Adam | Epoch 1 | Loss: 470.8691 | Validation Accuracy: 39.53%
Adam | Epoch 2 | Loss: 358.0287 | Validation Accuracy: 60.19%
Adam | Epoch 3 | Loss: 313.0530 | Validation Accuracy: 69.65%
Adam | Epoch 4 | Loss: 285.9501 | Validation Accuracy: 73.82%
Adam | Epoch 5 | Loss: 267.1241 | Validation Accuracy: 75.69%


In [10]:
y_pred_sgd = generate_predictions(model_sgd)

save_predictions(
    y_pred_sgd,
    "results/predicoes_augmentation_sgd.csv",
    experiment_name="CNN Data Augmentation SGD"
)

In [11]:
y_pred_momentum = generate_predictions(model_momentum)

save_predictions(
    y_pred_momentum,
    "results/predicoes_augmentation_momentum.csv",
    experiment_name="CNN Data Augmentation Momentum"
)

In [12]:
y_pred_adam = generate_predictions(model_adam)

save_predictions(
    y_pred_adam,
    "results/predicoes_augmentation_adam.csv",
    experiment_name="CNN Data Augmentation Adam"
)